In [2]:
import os
import librosa
import numpy as np
from tqdm import tqdm

DATA_PATH = '../Data/speech_commands/'
SAVE_PATH = '../Data/'

# Define classification targets
TARGET_WORDS = ['on', 'off', 'stop']
UNKNOWN_WORDS = ['right', 'left', 'up', 'down'] 

X = [] # List to store MFCC features
y = [] # List to store corresponding labels (0 to 4)

def get_mfcc(file_path):
    try:
        # Load audio file with 16000 Hz sample rate
        audio, sr = librosa.load(file_path, sr=16000)
        # Extract 13 MFCC features
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13, hop_length=512)
        # Average over time to compress features for constrained memory
        mfccs_processed = np.mean(mfccs.T, axis=0)
        return mfccs_processed
    except Exception as e:
        return None

print("Processing Target Words...")
for label_idx, word in enumerate(TARGET_WORDS):
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    # Limit the number of files to balance the dataset
    for file_name in tqdm(files[:1500], desc=f"Loading {word}"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(label_idx)

print("\nProcessing Unknown Words...")
# Set label index 4 for the 'Unknown' class
unknown_label_idx = len(TARGET_WORDS) 
for word in UNKNOWN_WORDS:
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    # Take fewer samples per unknown word to maintain overall class balance
    for file_name in tqdm(files[:300], desc=f"Loading {word} (Unknown)"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(unknown_label_idx)

# Convert lists to NumPy arrays
X = np.array(X)
y = np.array(y)

# Save features and labels for the training phase
np.save(os.path.join(SAVE_PATH, 'X_features.npy'), X)
np.save(os.path.join(SAVE_PATH, 'y_labels.npy'), y)

print(f"\nFeature extraction complete! Saved {len(X)} samples.")
print(f"Features shape: {X.shape}")

Processing Target Words...


Loading stop: 100%|██████████| 1500/1500 [00:23<00:00, 62.65it/s]



Processing Unknown Words...


Loading down (Unknown): 100%|██████████| 300/300 [00:04<00:00, 64.34it/s]


Feature extraction complete! Saved 5700 samples.
Features shape: (5700, 13)


RandomForest

In [29]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
# import m2cgen as m2c
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../Output'

# 1. Load the pre-processed features and labels
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Define the Optuna objective function for a TINY Random Forest model
def objective(trial):
    param = {
        # STRICT LIMITS: Keeping trees small for the MCU's RAM
        'n_estimators': trial.suggest_int('n_estimators', 10, 30), 
        'max_depth': trial.suggest_int('max_depth', 2, 10), 
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = RandomForestClassifier(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    return accuracy_score(y_test, preds)

# 3. Run the hyperparameter optimization
print("\nStarting Optuna optimization for a lightweight model...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\nBest parameters found:", study.best_params)
print(f"Best cross-validation accuracy: {study.best_value * 100:.2f}%")

# 4. Train the final Random Forest model
print("\nTraining the final Random Forest model...")
final_model = RandomForestClassifier(**study.best_params, random_state=42)
final_model.fit(X_train, y_train)

# Evaluate the final model
final_preds = final_model.predict(X_test)
print("\nClassification Report on Test Data:")

unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]

print(classification_report(y_test, final_preds, target_names=target_names))

# 5. Export the trained model parameters as lightweight arrays for MicroPython
print("\nExporting tree parameters (arrays) for memory-efficient Edge Deployment...")

trees_dump = []
for estimator in final_model.estimators_:
    tree = estimator.tree_
    
    # In Random Forest, tree.value stores the class counts at each node
    # We convert this to the actual predicted class index for leaf nodes
    node_classes = [int(np.argmax(val)) for val in tree.value]
    
    tree_data = {
        'left': tree.children_left.tolist(),
        'right': tree.children_right.tolist(),
        'feature': tree.feature.tolist(),
        # Round thresholds to 4 decimals to save storage space
        'threshold': [round(t, 4) for t in tree.threshold.tolist()], 
        'classes': node_classes
    }
    trees_dump.append(tree_data)

# Save to a Python file as a simple list of dictionaries
output_file = os.path.join(OUTPUT_PATH, 'model_data_randomForest.py')
with open(output_file, 'w') as f:
    f.write("# Auto-generated Random Forest parameters for Edge AI\n")
    f.write("TREES = [\n")
    for t in trees_dump:
        f.write(f"    {t},\n")
    f.write("]\n")

print(f"Model parameters successfully exported to: {output_file}")

[I 2026-06-28 15:36:50,570] A new study created in memory with name: no-name-f3095aef-2e81-4601-aed4-6ea58ef099b3
[I 2026-06-28 15:36:50,679] Trial 0 finished with value: 0.5780701754385965 and parameters: {'n_estimators': 27, 'max_depth': 9}. Best is trial 0 with value: 0.5780701754385965.


Loaded Data -> X shape: (5700, 13), y shape: (5700,)

Starting Optuna optimization for a lightweight model...


[I 2026-06-28 15:36:50,781] Trial 1 finished with value: 0.5359649122807018 and parameters: {'n_estimators': 26, 'max_depth': 4}. Best is trial 0 with value: 0.5780701754385965.
[I 2026-06-28 15:36:50,889] Trial 2 finished with value: 0.5736842105263158 and parameters: {'n_estimators': 26, 'max_depth': 9}. Best is trial 0 with value: 0.5780701754385965.
[I 2026-06-28 15:36:50,964] Trial 3 finished with value: 0.5578947368421052 and parameters: {'n_estimators': 15, 'max_depth': 9}. Best is trial 0 with value: 0.5780701754385965.
[I 2026-06-28 15:36:51,045] Trial 4 finished with value: 0.5701754385964912 and parameters: {'n_estimators': 13, 'max_depth': 9}. Best is trial 0 with value: 0.5780701754385965.
[I 2026-06-28 15:36:51,139] Trial 5 finished with value: 0.5289473684210526 and parameters: {'n_estimators': 23, 'max_depth': 4}. Best is trial 0 with value: 0.5780701754385965.
[I 2026-06-28 15:36:51,243] Trial 6 finished with value: 0.5763157894736842 and parameters: {'n_estimators': 2


Best parameters found: {'n_estimators': 30, 'max_depth': 10}
Best cross-validation accuracy: 61.14%

Training the final Random Forest model...

Classification Report on Test Data:
              precision    recall  f1-score   support

          on       0.57      0.67      0.61       300
         off       0.65      0.60      0.62       300
        stop       0.63      0.64      0.63       300
     unknown       0.62      0.53      0.57       240

    accuracy                           0.61      1140
   macro avg       0.61      0.61      0.61      1140
weighted avg       0.61      0.61      0.61      1140


Exporting tree parameters (arrays) for memory-efficient Edge Deployment...
Model parameters successfully exported to: ../Output\model_data_randomForest.py


LogisticRegression

In [30]:
import numpy as np
from sklearn.linear_model import LogisticRegression
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../Output'

# 1. Load the pre-processed features and labels
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Define the Optuna objective function for Logistic Regression
def objective(trial):
    param = {
        # 'C' is the inverse of regularization strength. 
        # Smaller values specify stronger limits/constraints on the weights.
        'C': trial.suggest_float('C', 1e-4, 1e2, log=True),
        
        # 'solver' is the algorithm used to optimize the weights
        'solver': trial.suggest_categorical('solver', ['lbfgs', 'newton-cg', 'saga']),
        
        # Max_iter is kept high to allow the mathematical solver to fully converge
        'max_iter': 2000, 
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = LogisticRegression(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    return accuracy_score(y_test, preds)

# 3. Run the hyperparameter optimization
print("\nStarting Optuna optimization for Logistic Regression...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\nBest parameters found:", study.best_params)
print(f"Best cross-validation accuracy: {study.best_value * 100:.2f}%")

# 4. Train the final model using the exact optimized parameters
print("\nTraining the final Logistic Regression model...")
final_model = LogisticRegression(**study.best_params, max_iter=2000, random_state=42)
final_model.fit(X_train, y_train)

# Evaluate the final model
final_preds = final_model.predict(X_test)
print("\nClassification Report on Test Data:")

# Dynamically extract classes
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]
print(classification_report(y_test, final_preds, target_names=target_names))

# 5. Export weights and intercepts to a tiny Python file
print("\nExporting model parameters (Weights & Intercepts) for Edge Deployment...")

weights = final_model.coef_.tolist()
intercepts = final_model.intercept_.tolist()

output_file = os.path.join(OUTPUT_PATH, 'model_data_regression.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Lightweight Model for Keyword Spotting\n\n")
    f.write(f"CLASSES = {target_names}\n\n")
    
    # We round the numbers to 4 decimal places to save even more space on the MCU
    f.write("WEIGHTS = [\n")
    for class_weights in weights:
        rounded_weights = [round(w, 4) for w in class_weights]
        f.write(f"    {rounded_weights},\n")
    f.write("]\n\n")
    
    rounded_intercepts = [round(i, 4) for i in intercepts]
    f.write(f"INTERCEPTS = {rounded_intercepts}\n")

print(f"Model successfully exported to: {output_file}")
print("Done! The model_data.py is now highly optimized and incredibly lightweight.")

[I 2026-06-28 15:43:01,258] A new study created in memory with name: no-name-b6f72796-de13-4c88-92b7-c6a99af59f8b


Loaded Data -> X shape: (5700, 13), y shape: (5700,)

Starting Optuna optimization for Logistic Regression...


c:\Users\User\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
[I 2026-06-28 15:43:01,569] Trial 0 finished with value: 0.5587719298245614 and parameters: {'C': 0.03357609695839051, 'solver': 'newton-cg'}. Best is trial 0 with value: 0.5587719298245614.
c:\Users\User\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
[I 2026-06-28 15:43:01,790] Trial 1 finished with value: 0.5447368421052632 and parameters: {'C': 0.009214388448335655, 'solver': 'saga'}. Best is trial 0 with value: 0.5587719298245614.
c:\Users\User\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs


Best parameters found: {'C': 0.0001104834739737416, 'solver': 'newton-cg'}
Best cross-validation accuracy: 56.67%

Training the final Logistic Regression model...

Classification Report on Test Data:
              precision    recall  f1-score   support

          on       0.54      0.58      0.56       300
         off       0.60      0.55      0.57       300
        stop       0.57      0.60      0.58       300
     unknown       0.56      0.53      0.54       240

    accuracy                           0.57      1140
   macro avg       0.57      0.56      0.57      1140
weighted avg       0.57      0.57      0.57      1140


Exporting model parameters (Weights & Intercepts) for Edge Deployment...
Model successfully exported to: ../Output\model_data_regression.py
Done! The model_data.py is now highly optimized and incredibly lightweight.


In [21]:
import sys
import os

sys.path.append(os.path.abspath('../Output'))

import model_data_regression

def predict_audio_class(mfcc_features):
    """
    Computes the dot product of features and weights for each class,
    adds the intercept, and returns the class with the highest score.
    """
    num_classes = len(model_data.INTERCEPTS)
    num_features = len(mfcc_features)
    
    scores = []
    
    # Calculate score for each class
    for i in range(num_classes):
        class_score = model_data.INTERCEPTS[i]
        
        # Dot product: multiply each feature by its corresponding weight
        for j in range(num_features):
            class_score += mfcc_features[j] * model_data.WEIGHTS[i][j]
            
        scores.append(class_score)
        
    # Find the index of the highest score (Argmax)
    best_class_idx = 0
    max_score = scores[0]
    for i in range(1, num_classes):
        if scores[i] > max_score:
            max_score = scores[i]
            best_class_idx = i
            
    return best_class_idx

# --- Example Usage on Board ---
# ['on', 'off', 'stop', 'uknown']
i = 16
features = X_test[i] # 13 features from mic
result_idx = predict_audio_class(features)
print("Predicted Word:", model_data.CLASSES[result_idx], "and actuall is:", y_test[i])

Predicted Word: stop and actuall is: 2


XGBoost

In [35]:
import numpy as np
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os
import json

DATA_PATH = '../Data/'
OUTPUT_PATH = '../Output'

# 1. Load the pre-processed features and labels
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Define Optuna objective for XGBoost
def objective(trial):
    param = {
        'objective': 'multi:softprob',
        'max_depth': trial.suggest_int('max_depth', 2, 5), 
        'n_estimators': trial.suggest_int('n_estimators', 10, 30), 
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = xgb.XGBClassifier(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    return accuracy_score(y_test, preds)

# 3. Optimize Hyperparameters
print("\nStarting Optuna optimization for XGBoost...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\nBest parameters found:", study.best_params)
print(f"Best cross-validation accuracy: {study.best_value * 100:.2f}%")

# 4. Train the Final XGBoost Model
print("\nTraining the final XGBoost model...")
best_params = study.best_params
best_params['objective'] = 'multi:softprob'
best_params['random_state'] = 42

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

# Evaluate the final model
final_preds = final_model.predict(X_test)
print("\nClassification Report on Test Data:")
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]
print(classification_report(y_test, final_preds, target_names=target_names))

# 5. CUSTOM EXPORT: Bypass m2cgen and extract trees directly
print("\nExporting XGBoost trees manually for ultra-lightweight Edge Deployment...")

booster = final_model.get_booster()
# Dump all trees to JSON format
trees_json = booster.get_dump(dump_format='json')

parsed_trees = []
for tree_str in trees_json:
    tree_dict = json.loads(tree_str)
    
    # Recursive function to find the maximum node ID to size our arrays
    def get_max_nodeid(node):
        max_id = node.get('nodeid', 0)
        if 'children' in node:
            for child in node['children']:
                max_id = max(max_id, get_max_nodeid(child))
        return max_id
        
    size = get_max_nodeid(tree_dict) + 1
    
    # Initialize flattened arrays for MicroPython
    features = [-1] * size
    thresholds = [0.0] * size
    lefts = [-1] * size
    rights = [-1] * size
    values = [0.0] * size
    
    # Recursive function to parse JSON into flat arrays
    def parse_node(node):
        nid = node['nodeid']
        if 'leaf' in node:
            values[nid] = round(node['leaf'], 4)
        else:
            # XGBoost features are formatted as 'f0', 'f1', etc. We extract the integer.
            feat_str = node['split']
            features[nid] = int(feat_str.replace('f', ''))
            thresholds[nid] = round(node['split_condition'], 4)
            lefts[nid] = node['yes']
            rights[nid] = node['no']
            for child in node['children']:
                parse_node(child)
                
    parse_node(tree_dict)
    
    parsed_trees.append({
        'features': features,
        'thresholds': thresholds,
        'left': lefts,
        'right': rights,
        'values': values
    })

# Save the parsed structure to a Python file
output_file = os.path.join(OUTPUT_PATH, 'model_data_xgb.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Custom XGBoost Export for MicroPython\n\n")
    f.write(f"CLASSES = {target_names}\n")
    f.write(f"NUM_CLASS = {len(target_names)}\n\n")
    f.write("TREES = [\n")
    for t in parsed_trees:
        f.write(f"    {t},\n")
    f.write("]\n")

print(f"XGBoost Model successfully exported to: {output_file}")

[I 2026-06-28 15:51:25,322] A new study created in memory with name: no-name-a2f8db2c-fcc4-43b5-b01b-f90dab892d13
[I 2026-06-28 15:51:25,396] Trial 0 finished with value: 0.5184210526315789 and parameters: {'max_depth': 3, 'n_estimators': 12, 'learning_rate': 0.043580105475497614}. Best is trial 0 with value: 0.5184210526315789.
[I 2026-06-28 15:51:25,472] Trial 1 finished with value: 0.5587719298245614 and parameters: {'max_depth': 3, 'n_estimators': 14, 'learning_rate': 0.22732922313923937}. Best is trial 1 with value: 0.5587719298245614.


Loaded Data -> X shape: (5700, 13), y shape: (5700,)

Starting Optuna optimization for XGBoost...


[I 2026-06-28 15:51:25,702] Trial 2 finished with value: 0.6008771929824561 and parameters: {'max_depth': 5, 'n_estimators': 26, 'learning_rate': 0.2314135206328629}. Best is trial 2 with value: 0.6008771929824561.
[I 2026-06-28 15:51:25,799] Trial 3 finished with value: 0.5333333333333333 and parameters: {'max_depth': 2, 'n_estimators': 29, 'learning_rate': 0.08339309776177732}. Best is trial 2 with value: 0.6008771929824561.
[I 2026-06-28 15:51:25,873] Trial 4 finished with value: 0.5236842105263158 and parameters: {'max_depth': 2, 'n_estimators': 22, 'learning_rate': 0.07736962285795046}. Best is trial 2 with value: 0.6008771929824561.
[I 2026-06-28 15:51:26,108] Trial 5 finished with value: 0.5815789473684211 and parameters: {'max_depth': 5, 'n_estimators': 29, 'learning_rate': 0.07023079022945707}. Best is trial 2 with value: 0.6008771929824561.
[I 2026-06-28 15:51:26,278] Trial 6 finished with value: 0.5798245614035088 and parameters: {'max_depth': 4, 'n_estimators': 27, 'learnin


Best parameters found: {'max_depth': 5, 'n_estimators': 26, 'learning_rate': 0.259231172603688}
Best cross-validation accuracy: 61.05%

Training the final XGBoost model...

Classification Report on Test Data:
              precision    recall  f1-score   support

          on       0.58      0.62      0.60       300
         off       0.64      0.60      0.62       300
        stop       0.62      0.67      0.65       300
     unknown       0.61      0.54      0.57       240

    accuracy                           0.61      1140
   macro avg       0.61      0.61      0.61      1140
weighted avg       0.61      0.61      0.61      1140


Exporting XGBoost trees manually for ultra-lightweight Edge Deployment...
XGBoost Model successfully exported to: ../Output\model_data_xgb.py


In [37]:
import sys
import os

sys.path.append(os.path.abspath('../Output'))

import model_data_xgb as model_data
def predict_audio_class(mfcc_features):
    """
    Traverses the exported XGBoost trees using the extracted MFCC features.
    Returns the index of the predicted class based on the highest raw score.
    """
    num_class = model_data.NUM_CLASS
    
    # Array to accumulate scores (raw logits) for each class
    class_scores = [0.0] * num_class
    
    # In multi-class XGBoost, trees are assigned to classes in a round-robin fashion.
    # Tree 0 -> Class 0, Tree 1 -> Class 1 ... Tree 4 -> Class 0, etc.
    for i, tree in enumerate(model_data.TREES):
        node = 0 # Start at the root of the tree
        
        # Traverse until a leaf is reached (marked by feature index -1)
        while tree['features'][node] != -1:
            feat_idx = tree['features'][node]
            
            # XGBoost splitting rule: strictly less than (<)
            if mfcc_features[feat_idx] < tree['thresholds'][node]:
                node = tree['left'][node]
            else:
                node = tree['right'][node]
                
        # Leaf reached: get the value and add it to the corresponding class score
        leaf_value = tree['values'][node]
        class_idx = i % num_class
        class_scores[class_idx] += leaf_value
        
    # Find the index of the highest score (Argmax)
    best_class = 0
    max_score = class_scores[0]
    for i in range(1, num_class):
        if class_scores[i] > max_score:
            max_score = class_scores[i]
            best_class = i
            
    return best_class

# --- Example Usage on Board ---
# ['on', 'off', 'stop', 'uknown']
i = 16
features = X_test[i] # 13 features from mic
result_idx = predict_audio_class(features)
print("Predicted Word:", model_data.CLASSES[result_idx], "and actuall is:", y_test[i])


Predicted Word: stop and actuall is: 2
